In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6bedc14d-89b5-41e0-9843-44552d8567e5;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 367ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
df_customers = spark.read.option('delimiter', ',') \
              .option('header', 'true') \
              .option('nullValue', 'NULL') \
              .csv('s3a://last-mile-optimization-raw/dataset-orders/olist_customers_dataset.csv')
df_customers.show()

26/04/04 22:10:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                   08775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [4]:
from pyspark.sql.functions import col, upper, regexp_replace

# Converter o tipo da coluna ‘customer_zip_code_prefix’ para String
df_customers = df_customers.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))

# Colocar nomes de cidade em letra maiúscula
df_customers = df_customers.withColumn('customer_city', upper(col('customer_city')))

# Remover acentos
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[áàâãä]', 'a'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[éèêë]', 'e'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[íìîï]', 'i'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[óòôõö]', 'o'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[úùûü]', 'u'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), 'ç', 'c'))


# Retirar todos estados que não sejam SP
df_customers_sp = df_customers.filter(col('customer_state') == 'SP')

# Remover a coluna customer_state
df_customers_sp = df_customers_sp.drop("customer_state")

df_customers_sp.show()

+--------------------+--------------------+------------------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|
+--------------------+--------------------+------------------------+--------------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              FRANCA|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|SAO BERNARDO DO C...|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           SAO PAULO|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                   08775|     MOGI DAS CRUZES|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            CAMPINAS|
|fd826e7cf63160e53...|addec96d2e059c80c...|                   04534|           SAO PAULO|
|b2d1536598b73a9ab...|918dc87cd72cd9f6e...|                   18682|    LENCOIS PAULISTA|
|eabebad39a88bb6f5...|295c05e81917928d7...|                   05704|           SAO PAULO|
|206f3129c

In [5]:
df_customers_sp.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/customers_cleaned_dataset.csv')

spark.stop()

26/04/04 22:10:46 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 22:10:46 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
